# 01 — Data Exploration

This notebook profiles the normalized Retailrocket funnel dataset and prepares observations for data quality and funnel analysis.

## Coverage
- Environment and library setup
- Data loading and schema inspection
- Numeric and categorical summaries
- Missing values and distributions
- Funnel and segment previews
- Exported exploration summary

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "ecommerce_funnel_events.csv"
REPORTS_PATH = ROOT / "reports"
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Normalized dataset not found at {DATA_PATH}. "
        "Run python scripts/ingest_retailrocket.py or generate the sample dataset first."
    )

df = pd.read_csv(DATA_PATH)
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"], utc=True, errors="coerce")

print("Source:", DATA_PATH)
print("Dimensions:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
display(df.tail())

structure = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100).round(2),
    "unique_count": df.nunique(dropna=True),
})
display(structure)

numeric = df.select_dtypes(include=np.number)
if not numeric.empty:
    display(numeric.describe().T)
else:
    print("No numeric columns found.")

categorical = df.select_dtypes(include=["object", "category"]).columns
for column in categorical:
    frequencies = df[column].fillna("<MISSING>").value_counts(dropna=False)
    summary = pd.DataFrame({
        "count": frequencies,
        "proportion": (frequencies / len(df)).round(4),
    })
    print(f"\n{column} — {df[column].nunique(dropna=True)} unique values")
    display(summary.head(20))

if numeric.shape[1] >= 2:
    correlation = numeric.corr()
    fig, ax = plt.subplots(figsize=(8, 5))
    image = ax.imshow(correlation, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=45, ha="right")
    ax.set_yticks(range(len(correlation.columns)), correlation.columns)
    for row in range(len(correlation.columns)):
        for column in range(len(correlation.columns)):
            ax.text(column, row, f"{correlation.iloc[row, column]:.2f}", ha="center", va="center")
    fig.colorbar(image, ax=ax)
    ax.set_title("Numeric Feature Correlations")
    plt.tight_layout()
    plt.savefig(REPORTS_PATH / "exploration_correlations.png", dpi=150)
    plt.show()

stage_counts = df.groupby("stage")["user_id"].nunique().reindex(["product_view", "cart", "purchase"])
fig, ax = plt.subplots(figsize=(8, 5))
stage_counts.plot(kind="bar", ax=ax, color=["#4C78A8", "#F2CF5B", "#59A14F"])
ax.set_title("Unique Users by Funnel Stage")
ax.set_xlabel("Funnel stage")
ax.set_ylabel("Unique users")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORTS_PATH / "exploration_funnel.png", dpi=150)
plt.show()

exploration_summary = pd.DataFrame({
    "metric": ["rows", "columns", "unique_users", "valid_timestamps", "duplicate_rows"],
    "value": [len(df), len(df.columns), df["user_id"].nunique(), df["event_timestamp"].notna().sum(), df.duplicated().sum()],
})
exploration_summary.to_csv(REPORTS_PATH / "exploration_summary.csv", index=False)
structure.to_csv(REPORTS_PATH / "exploration_schema_quality.csv")
display(exploration_summary)